In [86]:
import os
import mlflow
import pickle
import tempfile
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from mlflow.tracking import MlflowClient
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from urllib.request import urlretrieve
from pathlib import Path

This notebook contains the code to:
1. load a data set
2. train a model
3. validate the model
4. test the model on a new data set
5. promote a model

In [2]:
MLFLOW_TRACKING_URI = 'http://localhost:5000'
EXPERIMENT = 'nyc-taxi-end-to-end'

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)

<Experiment: artifact_location='mlflow-artifacts:/6', creation_time=1775304067127, experiment_id='6', last_update_time=1775304067127, lifecycle_stage='active', name='nyc-taxi-end-to-end', tags={}, workspace='default'>

In [4]:
client.create_dataset(name=EXPERIMENT)

<EvaluationDataset: experiment_ids=[], profile=None, records=[], schema=None, source=<mlflow.data.evaluation_dataset_source.EvaluationDatasetSource object at 0x301001e80>>

In [9]:
training_dataset_name = 'green_tripdata_2021-01'
training_dataset_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet'
validation_dataset_name = 'green_tripdata_2021-02'
validation_dataset_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet'

In [12]:
def load_parquet_dataset(name, url):
    path = f'data/{name}.parquet'
    if not(os.path.exists(path)):
        urlretrieve(url, path)
    
    df = pd.read_parquet(path)
    return df

In [13]:
training_df = load_parquet_dataset(training_dataset_name, training_dataset_url)
training_df

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-01-01 00:15:56,2021-01-01 00:19:52,N,1.0,43,151,1.0,1.01,5.50,0.50,0.5,0.00,0.00,None,0.3,6.80,2.0,1.0,0.00
1,2,2021-01-01 00:25:59,2021-01-01 00:34:44,N,1.0,166,239,1.0,2.53,10.00,0.50,0.5,2.81,0.00,None,0.3,16.86,1.0,1.0,2.75
2,2,2021-01-01 00:45:57,2021-01-01 00:51:55,N,1.0,41,42,1.0,1.12,6.00,0.50,0.5,1.00,0.00,None,0.3,8.30,1.0,1.0,0.00
3,2,2020-12-31 23:57:51,2021-01-01 00:04:56,N,1.0,168,75,1.0,1.99,8.00,0.50,0.5,0.00,0.00,None,0.3,9.30,2.0,1.0,0.00
4,2,2021-01-01 00:16:36,2021-01-01 00:16:40,N,2.0,265,265,3.0,0.00,-52.00,0.00,-0.5,0.00,0.00,None,-0.3,-52.80,3.0,1.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76513,2,2021-01-31 21:38:00,2021-01-31 22:16:00,None,NaN,81,90,NaN,17.63,56.23,2.75,0.0,0.00,6.12,None,0.3,65.40,NaN,NaN,NaN
76514,2,2021-01-31 22:43:00,2021-01-31 23:21:00,None,NaN,35,213,NaN,18.36,46.66,0.00,0.0,12.20,6.12,None,0.3,65.28,NaN,NaN,NaN
76515,2,2021-01-31 22:16:00,2021-01-31 22:27:00,None,NaN,74,69,NaN,2.50,18.95,2.75,0.0,0.00,0.00,None,0.3,22.00,NaN,NaN,NaN
76516,2,2021-01-31 23:10:00,2021-01-31 23:37:00,None,NaN,168,215,NaN,14.48,48.87,2.75,0.0,0.00,6.12,None,0.3,58.04,NaN,NaN,NaN


In [14]:
validation_df = load_parquet_dataset(validation_dataset_name, validation_dataset_url)
validation_df

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-02-01 00:34:03,2021-02-01 00:51:58,N,1.0,130,205,5.0,3.66,14.00,0.50,0.5,10.00,0.0,None,0.3,25.30,1.0,1.0,0.00
1,2,2021-02-01 00:04:00,2021-02-01 00:10:30,N,1.0,152,244,1.0,1.10,6.50,0.50,0.5,0.00,0.0,None,0.3,7.80,2.0,1.0,0.00
2,2,2021-02-01 00:18:51,2021-02-01 00:34:06,N,1.0,152,48,1.0,4.93,16.50,0.50,0.5,0.00,0.0,None,0.3,20.55,2.0,1.0,2.75
3,2,2021-02-01 00:53:27,2021-02-01 01:11:41,N,1.0,152,241,1.0,6.70,21.00,0.50,0.5,0.00,0.0,None,0.3,22.30,2.0,1.0,0.00
4,2,2021-02-01 00:57:46,2021-02-01 01:06:44,N,1.0,75,42,1.0,1.89,8.50,0.50,0.5,2.45,0.0,None,0.3,12.25,1.0,1.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64567,2,2021-02-28 22:19:00,2021-02-28 22:29:00,None,NaN,129,7,NaN,2.63,10.04,0.00,0.0,0.00,0.0,None,0.3,10.34,NaN,NaN,NaN
64568,2,2021-02-28 23:18:00,2021-02-28 23:27:00,None,NaN,116,166,NaN,1.87,8.33,0.00,0.0,1.89,0.0,None,0.3,10.52,NaN,NaN,NaN
64569,2,2021-02-28 23:44:00,2021-02-28 23:58:00,None,NaN,74,151,NaN,2.40,12.61,0.00,0.0,0.00,0.0,None,0.3,12.91,NaN,NaN,NaN
64570,2,2021-02-28 23:07:00,2021-02-28 23:14:00,None,NaN,42,42,NaN,1.11,11.95,2.75,0.0,0.00,0.0,None,0.3,15.00,NaN,NaN,NaN


In [16]:
def preprocess_dataset(df):
    df = df[df.trip_type == 2]
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [17]:
training_df = preprocess_dataset(training_df)
training_df

/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/ipykernel_2214/82595708.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/ipykernel_2214/82595708.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration
30,2,2021-01-01 00:35:29,2021-01-01 00:55:15,N,5.0,74,247,1.0,3.64,13.0,...,0.0,0.0,0.0,None,0.3,13.3,2.0,2.0,0.0,19.766667
53,2,2021-01-01 01:54:51,2021-01-01 02:15:35,N,5.0,74,94,1.0,5.82,18.0,...,0.0,0.0,0.0,None,0.3,18.3,2.0,2.0,0.0,20.733333
69,2,2021-01-01 02:42:49,2021-01-01 02:50:59,N,5.0,136,241,1.0,0.57,9.0,...,0.0,0.0,0.0,None,0.3,9.3,2.0,2.0,0.0,8.166667
88,2,2021-01-01 04:52:02,2021-01-01 05:05:01,N,5.0,247,75,1.0,3.43,15.0,...,0.0,0.0,0.0,None,0.3,15.3,2.0,2.0,0.0,12.983333
96,2,2021-01-01 05:52:43,2021-01-01 05:58:02,N,5.0,7,7,1.0,0.65,50.0,...,0.0,7.0,0.0,None,0.3,57.3,1.0,2.0,0.0,5.316667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40324,2,2021-01-31 18:42:52,2021-01-31 19:08:44,N,5.0,69,213,2.0,5.40,16.0,...,0.0,0.0,0.0,None,0.3,16.3,2.0,2.0,0.0,25.866667
40352,1,2021-01-31 19:32:22,2021-01-31 19:50:16,N,5.0,82,129,2.0,2.40,0.0,...,0.0,0.0,0.0,None,0.0,0.0,2.0,2.0,0.0,17.900000
40363,2,2021-01-31 19:26:20,2021-01-31 19:35:47,N,5.0,167,248,2.0,1.31,10.0,...,0.0,0.0,0.0,None,0.3,10.3,2.0,2.0,0.0,9.450000
40364,2,2021-01-31 19:44:54,2021-01-31 20:19:56,N,5.0,147,147,2.0,6.97,30.0,...,0.0,0.0,0.0,None,0.3,30.3,2.0,2.0,0.0,35.033333


In [18]:
validation_df = preprocess_dataset(validation_df)
validation_df

/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/ipykernel_2214/82595708.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/ipykernel_2214/82595708.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration
5,2,2021-02-01 00:33:03,2021-02-01 00:40:54,N,5.0,197,219,1.0,3.30,15.0,...,0.0,0.76,0.00,None,0.3,16.06,1.0,2.0,0.00,7.850000
9,2,2021-02-01 02:56:55,2021-02-01 02:58:38,N,5.0,78,78,1.0,0.00,24.0,...,0.0,0.00,0.00,None,0.3,24.30,2.0,2.0,0.00,1.716667
114,2,2021-02-01 19:01:09,2021-02-01 19:11:20,N,5.0,75,74,1.0,1.80,15.0,...,0.0,0.00,0.00,None,0.3,15.30,1.0,2.0,0.00,10.183333
125,2,2021-02-01 22:57:12,2021-02-01 23:07:36,N,5.0,223,92,1.0,4.98,50.0,...,0.0,0.00,0.00,None,0.3,50.30,2.0,2.0,0.00,10.400000
150,2,2021-02-02 06:31:38,2021-02-02 06:50:10,N,5.0,223,100,1.0,5.64,50.0,...,0.0,13.26,0.00,None,0.3,66.31,1.0,2.0,2.75,18.533333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35391,2,2021-02-28 20:15:35,2021-02-28 20:32:59,N,5.0,243,235,2.0,2.32,16.0,...,0.0,0.00,0.00,None,0.3,16.30,2.0,2.0,0.00,17.400000
35423,2,2021-02-28 21:54:17,2021-02-28 22:04:58,N,5.0,247,69,2.0,1.51,18.0,...,0.0,0.00,0.00,None,0.3,18.30,2.0,2.0,0.00,10.683333
35446,2,2021-02-28 22:07:07,2021-02-28 22:25:11,N,5.0,69,78,2.0,3.57,12.0,...,0.0,0.00,0.00,None,0.3,12.30,2.0,2.0,0.00,18.066667
35447,2,2021-02-28 22:41:32,2021-02-28 23:08:58,N,5.0,69,74,2.0,5.06,17.0,...,0.0,0.00,0.00,None,0.3,17.30,2.0,2.0,0.00,27.433333


In [19]:
def load_dataset(df, dataset_name: str, source: str, predictions: str):
    dataset = mlflow.data.from_pandas(
        df,
        name=dataset_name,
        source=source,
        predictions=predictions
    )
    return dataset

In [20]:
training_dataset = load_dataset(
    df=training_df,
    dataset_name=training_dataset_name,
    source=training_dataset_url,
    predictions='duration'
)
training_dataset

In [21]:
validation_dataset = load_dataset(
    df=validation_df,
    dataset_name=validation_dataset_name,
    source=validation_dataset_url,
    predictions='duration'
)
validation_dataset

In [22]:
def vectorize_dataset(df, dv):
    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']
    
    dicts = df[categorical + numerical].to_dict(orient='records')
    
    X = None
    if dv:
        X = dv.transform(dicts)
    else:
        dv = DictVectorizer()
        X = dv.fit_transform(dicts)
    
    target = 'duration'
    
    y = df[target].values
    
    return X, y, dv

In [23]:
X_training, y_training, dv = vectorize_dataset(training_dataset.df, None)
X_validation, y_validation, _ = vectorize_dataset(validation_dataset.df, dv)

In [73]:
def train(train_dataset, val_dataset, dv):
    with mlflow.start_run():
        mlflow.log_input(train_dataset, context='training')
        mlflow.log_input(val_dataset, context='validation')
        
        mlflow.set_tag('developer', 'daniel')
    
        alpha = 0.1
        mlflow.log_param('alpha', alpha)
    
        X_train, y_train, dv = vectorize_dataset(training_dataset.df, None)
        X_val, y_val, _ = vectorize_dataset(validation_dataset.df, dv)

        lr = Lasso(alpha)
        lr.fit(X_train, y_train)
        
        y_pred = lr.predict(X_val)
        
        with tempfile.TemporaryDirectory() as tmp_dir:
            dictvectorizer_path = Path(tmp_dir, 'dictvectorizer.pkl')
            with open(dictvectorizer_path, 'wb') as dictvectorizer_file:
                pickle.dump(dv, dictvectorizer_file)
            # dictvectorizer_path.write_bytes(dv)
            model_info = mlflow.sklearn.log_model(lr, extra_files=[dictvectorizer_path])
        
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric('rmse', rmse)

        return model_info

In [74]:
model_info = train(training_dataset, validation_dataset, dv)
model_name = model_info.name
model_info.name, model_info.model_id

2026/04/11 10:35:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run bright-stag-544 at: http://localhost:5000/#/experiments/6/runs/1c992af7f0e046e398ae96e066f7429c
🧪 View experiment at: http://localhost:5000/#/experiments/6


('omniscient-roo-205', 'm-4506ae35bee64263b823e6f5f514712b')

In [56]:
mlflow.search_logged_models(filter_string=f"name = '{model_name}'")

,artifact_location,creation_timestamp,experiment_id,last_updated_timestamp,metrics,model_id,model_type,name,params,source_run_id,status,status_message,tags
0,mlflow-artifacts:/6/models/m-53200d5155e44fb88...,1775910848850,6,1775910851476,[],m-53200d5155e44fb8845ac2dba8651270,,gifted-finch-317,{'alpha': '0.1'},c4eb972e0b7540d899a6e094719920c2,READY,,"{'mlflow.source.name': 'end-to-end.ipynb', 'ml..."


In [58]:
model = mlflow.get_logged_model(model_info.model_id)
model

LoggedModel(artifact_location='mlflow-artifacts:/6/models/m-53200d5155e44fb8845ac2dba8651270/artifacts', creation_timestamp=1775910848850, experiment_id='6', last_updated_timestamp=1775910851476, model_id='m-53200d5155e44fb8845ac2dba8651270', model_type='', model_uri='models:/m-53200d5155e44fb8845ac2dba8651270', name='gifted-finch-317', source_run_id='c4eb972e0b7540d899a6e094719920c2', status=<LoggedModelStatus.READY: 'READY'>, status_message='')

In [59]:
evaluation_dataset_name = 'green_tripdata_2021-03'
evaluation_dataset_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-03.parquet'

In [60]:
evaluation_df = load_parquet_dataset(evaluation_dataset_name, evaluation_dataset_url)
evaluation_df

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2,2021-03-01 00:05:42,2021-03-01 00:14:03,N,1.0,83,129,1.0,1.56,7.50,0.50,0.5,0.00,0.0,None,0.3,8.80,1.0,1.0,0.0
1,2,2021-03-01 00:21:03,2021-03-01 00:26:17,N,1.0,243,235,1.0,0.96,6.00,0.50,0.5,0.00,0.0,None,0.3,7.30,2.0,1.0,0.0
2,2,2021-03-01 00:02:06,2021-03-01 00:22:26,N,1.0,75,242,1.0,9.93,28.00,0.50,0.5,2.00,0.0,None,0.3,31.30,1.0,1.0,0.0
3,2,2021-03-01 00:24:03,2021-03-01 00:31:43,N,1.0,242,208,1.0,2.57,9.50,0.50,0.5,0.00,0.0,None,0.3,10.80,2.0,1.0,0.0
4,1,2021-03-01 00:11:10,2021-03-01 00:14:46,N,1.0,41,151,1.0,0.80,5.00,0.50,0.5,1.85,0.0,None,0.3,8.15,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83822,2,2021-03-31 22:07:00,2021-03-31 22:13:00,None,NaN,41,75,NaN,1.48,8.46,0.00,0.0,1.44,0.0,None,0.3,10.20,NaN,NaN,NaN
83823,2,2021-03-31 22:56:00,2021-03-31 23:13:00,None,NaN,95,95,NaN,0.09,54.25,2.75,0.0,0.00,0.0,None,0.3,57.30,NaN,NaN,NaN
83824,2,2021-03-31 22:36:00,2021-03-31 22:45:00,None,NaN,95,95,NaN,0.66,8.11,0.00,0.0,0.00,0.0,None,0.3,8.41,NaN,NaN,NaN
83825,2,2021-03-31 23:35:00,2021-04-01 00:00:00,None,NaN,37,14,NaN,9.58,36.83,2.75,0.0,0.00,0.0,None,0.3,39.88,NaN,NaN,NaN


In [75]:
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    mlflow.artifacts.download_artifacts(artifact_uri=model_info.model_uri, dst_path=tmp_path)
    with open(Path(tmp_dir, 'extra_files', 'dictvectorizer.pkl'), 'rb') as dictvectorizer_file:
       dv = pickle.load(dictvectorizer_file)

/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/extra_files
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/python_env.yaml
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/requirements.txt
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/MLmodel
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/model.pkl
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/registered_model_meta
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/conda.yaml
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/tmp9ge2vts9/extra_files/dictvectorizer.pkl


In [81]:
def evaluate(model_uri, evaluation_df, evaluation_dataset_name, evaluation_dataset_url):
    evaluation_df = preprocess_dataset(evaluation_df)
    evaluation_dataset = load_dataset(
        df=evaluation_df,
        dataset_name=evaluation_dataset_name,
        source=evaluation_dataset_url,
        predictions='duration'
    )

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        mlflow.artifacts.download_artifacts(artifact_uri=model_uri, dst_path=tmp_path)
        with open(Path(tmp_dir, 'extra_files', 'dictvectorizer.pkl'), 'rb') as dictvectorizer_file:
           dv = pickle.load(dictvectorizer_file)
        with open(Path(tmp_dir, 'model.pkl'), 'rb') as model_file:
           model = pickle.load(model_file)

    X_validation, y_validation, _ = vectorize_dataset(evaluation_dataset.df, dv)

    X_eval, y_eval, _ = vectorize_dataset(evaluation_dataset.df, dv)

    lr = model
    
    y_pred = lr.predict(X_eval)
    
    rmse = root_mean_squared_error(y_eval, y_pred)
    return rmse

In [82]:
rmse = evaluate(model_info.model_uri, evaluation_df, evaluation_dataset_name, evaluation_dataset_url)
rmse

/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/ipykernel_2214/82595708.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
/var/folders/8b/0kqkw5_d00b0v_ggpz2p7gyc0000gn/T/ipykernel_2214/82595708.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)


9.031497517835964

In [83]:
model_version = mlflow.register_model(model_info.model_uri, EXPERIMENT, tags={'rmse': rmse})
model_version

Registered model 'nyc-taxi-end-to-end' already exists. Creating a new version of this model...
2026/04/11 10:53:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-end-to-end, version 2
Created version '2' of model 'nyc-taxi-end-to-end'.


<ModelVersion: aliases=[], creation_timestamp=1775915603620, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1775915603620, metrics=None, model_id=None, name='nyc-taxi-end-to-end', params=None, run_id='1c992af7f0e046e398ae96e066f7429c', run_link='', source='models:/m-4506ae35bee64263b823e6f5f514712b', status='READY', status_message=None, tags={'rmse': '9.031497517835964'}, user_id='', version='2', workspace='default'>

In [88]:
client.set_registered_model_alias(EXPERIMENT, 'staging', model_version.version)
client.update_model_version(EXPERIMENT, model_version.version, f'Promoted to staging on {datetime.today().date()}')
client.set_model_version_tag(EXPERIMENT, model_version.version, 'staging-promotion-date', datetime.today().date())